# Online range evaluation (artifact CSV)

Loads **combo-range probabilities** from `artifacts/online_range_history.csv` (see `runner.py session-split` / ``--range-csv-out``). Joins each row to the Pluribus ``.phh`` via `(session, hand_number)` to attach **cumulative board**, **target hole cards**, and **p1–p6** seats.

**Brier**: preflop rows use 1,326 → **169** collapse; postflop rows use full **1,326** keys. **Expected strength** uses Monte Carlo under the row distribution (faster than enumerating all combos); set `STRENGTH_MC_SAMPLES` or a row cap while iterating.

Run from the repo root with the project virtualenv (`.venv`) as the kernel.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path.cwd().resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from utils.eval.online_csv import (
    add_calibration_columns,
    combo_probability_columns,
    enrich_online_range_dataframe,
)


In [2]:
# --- configure ---
CSV_PATH = REPO / "artifacts" / "online_range_history.csv"
PLURIBUS_ROOT = REPO / "pluribus"
STRENGTH_MC_SAMPLES = 96
RNG_SEED = 0
VERBOSE = True  # stdout progress from enrich + calibration
PROGRESS_EVERY_ENRICH = 100
PROGRESS_EVERY_CALIB = 50
# set to None for all rows (664 rows × MC may take a few minutes)
MAX_ROWS: int | None = None


In [3]:
df_raw = pd.read_csv(CSV_PATH)
combo_cols = combo_probability_columns(df_raw)
if MAX_ROWS is not None:
    df_raw = df_raw.iloc[: MAX_ROWS].copy()

df = enrich_online_range_dataframe(
    df_raw,
    PLURIBUS_ROOT,
    verbose=VERBOSE,
    progress_every=PROGRESS_EVERY_ENRICH,
)
rng = np.random.default_rng(RNG_SEED)
df = add_calibration_columns(
    df,
    combo_cols,
    strength_mc_samples=STRENGTH_MC_SAMPLES,
    strength_rng=rng,
    verbose=VERBOSE,
    progress_every=PROGRESS_EVERY_CALIB,
)
df


[eval] enrich_online_range_dataframe: 663 rows | pluribus_root=/Users/kevinliu/Desktop/bayesian poker/pluribus
[eval] enrich … row 1/663
[eval] enrich … row 100/663
[eval] enrich … row 200/663
[eval] enrich … row 300/663
[eval] enrich … row 400/663
[eval] enrich … row 500/663
[eval] enrich … row 600/663
[eval] enrich … row 663/663
[eval] enrich_online_range_dataframe: done (152 unique session/hand hands cached)
[eval] add_calibration_columns: 663 rows | strength_mc_samples=96
[eval] calibration … row 1/663
[eval] calibration … row 50/663
[eval] calibration … row 100/663
[eval] calibration … row 150/663
[eval] calibration … row 200/663
[eval] calibration … row 250/663
[eval] calibration … row 300/663
[eval] calibration … row 350/663
[eval] calibration … row 400/663
[eval] calibration … row 450/663
[eval] calibration … row 500/663
[eval] calibration … row 550/663
[eval] calibration … row 600/663
[eval] calibration … row 650/663
[eval] calibration … row 663/663
[eval] add_calibration_colu

,session,hand_number,street,observer,target,3s2s,4s2s,5s2s,6s2s,7s2s,...,p2,p3,p4,p5,p6,brier,expected_made_pct,expected_draw,actual_made_pct,actual_draw
0,30,0,pre-flop,Gogo,Pluribus,0.000823,0.000830,0.000000,0.000847,0.000853,...,Gogo,Budd,Eddie,Bill,Pluribus,1.002092,NaN,NaN,NaN,NaN
1,30,0,pre-flop,Pluribus,Gogo,0.000790,0.000804,0.000819,0.000837,0.000850,...,Gogo,Budd,Eddie,Bill,Pluribus,0.986951,NaN,NaN,NaN,NaN
2,30,1,pre-flop,Gogo,Pluribus,0.000830,0.000838,0.000848,0.000860,0.000867,...,Budd,Eddie,Bill,Pluribus,MrWhite,0.985237,NaN,NaN,NaN,NaN
3,30,1,pre-flop,Pluribus,Gogo,0.000741,0.000725,0.000707,0.000684,0.000670,...,Budd,Eddie,Bill,Pluribus,MrWhite,0.989704,NaN,NaN,NaN,NaN
4,30,2,pre-flop,Gogo,Pluribus,0.000823,0.000831,0.000840,0.000850,0.000857,...,Eddie,Bill,Pluribus,MrWhite,Gogo,0.986352,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658,35,71,river,Gogo,Pluribus,0.001466,0.001659,0.001397,0.001354,0.001174,...,MrWhite,Gogo,Budd,Eddie,Bill,0.998885,0.508859,0.000000,0.680303,0.000000
659,35,71,pre-flop,Pluribus,Gogo,0.000722,0.000703,0.000680,0.000654,0.000638,...,MrWhite,Gogo,Budd,Eddie,Bill,0.964119,NaN,NaN,NaN,NaN
660,35,71,flop,Pluribus,Gogo,0.001031,0.001175,0.000727,0.000929,0.000706,...,MrWhite,Gogo,Budd,Eddie,Bill,0.997164,0.624513,0.151515,0.585106,0.272727
661,35,71,turn,Pluribus,Gogo,0.001126,0.001012,0.000794,0.001015,0.000737,...,MrWhite,Gogo,Budd,Eddie,Bill,0.996817,0.559732,0.000000,0.569082,0.000000


In [4]:
display(df.groupby("street", sort=False)["brier"].agg(["mean", "count"]))

post = df[df["community_cards"].str.len() >= 6]
display(
    post.groupby("street", sort=False)[
        ["expected_made_pct", "actual_made_pct", "expected_draw", "actual_draw"]
    ].mean(numeric_only=True)
)


,mean,count
street,,
pre-flop,0.990816,299
flop,0.998810,168
turn,0.998748,118
river,0.998655,78


,expected_made_pct,actual_made_pct,expected_draw,actual_draw
street,,,,
flop,0.542407,0.554428,0.070673,0.072240
turn,0.546179,0.557373,0.059137,0.053544
river,0.546402,0.511765,0.000000,0.000000


In [ ]:
display(
    post.groupby("street", sort=False).apply(
        lambda g: pd.Series(
            {
                "made_mse": np.mean(
                    (g["expected_made_pct"] - g["actual_made_pct"]) ** 2
                ),
                "draw_mse": np.mean((g["expected_draw"] - g["actual_draw"]) ** 2),
            }
        )
    )
)